# Data Cleaning

In [3]:
import pandas as pd 

### **Spotting data uncleanliness is just as important as fixing it**, and beginners often miss this step.

Here’s a **structured approach you can follow** to **spot unclean data** systematically:

#### ✅ 1. **Load the Dataset**

```python
import pandas as pd

df = pd.read_csv("patients.csv")
```

In [4]:
# Load the CSV file
df = pd.read_csv('C:\\Users\\frank\\data_science_project_advance\\datasets\\patients.csv')

#### 🔍 2. **Start with Basic Exploration**

Use `.info()`, `.describe()`, `.head()`, `.tail()`, `.sample()`

```python
print(df.info())       # Shows column types and missing values
print(df.describe())   # Stats for numeric columns
print(df.head())       # Preview first few rows
```

✅ **What to Look For**:

* `Non-null count` < total rows → **Missing Values**
* Weird data types (e.g., object for numbers)
* Large difference between min and max → **Outliers**

In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1995 entries, 0 to 1994
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           1995 non-null   object 
 1   Age            1995 non-null   object 
 2   BloodPressure  1782 non-null   float64
 3   Diagnosis      1995 non-null   object 
dtypes: float64(1), object(3)
memory usage: 62.5+ KB
None


Age shouldnt be an object type. It should an integer type. 
The fact that it's `object` means it likely contains **non-numeric values**, such as:

* Missing values stored as strings like `"unknown"`, `"N/A"`, or empty strings `""`
* Inconsistent formats, like `"30 years"` instead of `30`
* Words instead of numbers, like `"forty"`

### ✅ **How to Fix It**

Here’s a step-by-step approach in code:

In [7]:
# Check unique values to identify bad entries
print(df['Age'].unique())

['23' '49' '19' '74' '85' '58' '81' '84' '69' '77' '30' '72' '88' '18'
 '38' 'thirty' '89' '42' '87' '73' '50' '61' '32' '70' '59' '36' '51' '82'
 '78' '47' '66' '24' '45' '76' 'twenty' '46' '34' '75' '41' '35' '27' '65'
 '80' '20' '55' '21' '37' '43' '53' '57' '29' '25' '44' '64' '67' '48'
 '63' '62' '31' '68' '71' '54' '79' '39' '40' '60' '56' '28' '26' '83'
 '33' '52' '86' '22' 'fifty' 'sixty' 'forty']


As suspected, the `Age` column is a mix of **numeric strings** (e.g., `'23'`, `'49'`) and **word-based numbers** like:

* `'thirty'`
* `'twenty'`
* `'forty'`
* `'fifty'`
* `'sixty'`

These word-based entries are the reason the column is stored as `object` instead of a numeric type.

---

### ✅ Here's How to Clean and Convert `Age` Properly

You can map the word-based ages to their numeric equivalents and then convert the column to `float` or `int`.

In [8]:
# Mapping for word-based numbers
word_to_num = {
    'twenty': 20,
    'thirty': 30,
    'forty': 40,
    'fifty': 50,
    'sixty': 60
}

# Replace word-based entries with corresponding numeric values
df['Age'] = df['Age'].replace(word_to_num)

# Now convert the entire column to numeric (this will coerce any remaining bad values to NaN)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Confirm conversion
print(df.dtypes)
print(df['Age'].unique())

Name              object
Age                int64
BloodPressure    float64
Diagnosis         object
dtype: object
[23 49 19 74 85 58 81 84 69 77 30 72 88 18 38 89 42 87 73 50 61 32 70 59
 36 51 82 78 47 66 24 45 76 20 46 34 75 41 35 27 65 80 55 21 37 43 53 57
 29 25 44 64 67 48 63 62 31 68 71 54 79 39 40 60 56 28 26 83 33 52 86 22]


In [9]:
print(df.describe())   # Stats for numeric columns

               Age  BloodPressure
count  1995.000000    1782.000000
mean     53.249123     135.888328
std      20.507322      30.173937
min      18.000000      30.000000
25%      36.000000     112.000000
50%      53.000000     135.000000
75%      71.000000     157.000000
max      89.000000     400.000000


Based on the summary statistics, the `BloodPressure` column **contains outliers**.

---

### 🩺 **What Is the Ideal Range for Blood Pressure?**

According to medical guidelines (e.g., WHO, American Heart Association), **normal adult blood pressure** is generally considered to be:

| Category             | Systolic (upper) | Diastolic (lower) |
| -------------------- | ---------------- | ----------------- |
| Normal               | **< 120**        | **< 80**          |
| Elevated             | **120–129**      | **< 80**          |
| Hypertension Stage 1 | **130–139**      | **80–89**         |
| Hypertension Stage 2 | **140+**         | **90+**           |
| Hypertensive Crisis  | **180+**         | **120+**          |

In practice, **a realistic/physiological range** for systolic blood pressure values is roughly:

> **🟢 Normal range: 90–180 mmHg**
> Values **below 80** or **above 200** are often suspect (either outliers or entry errors).

---

### 🚨 The Stats

| Stat     | Value                  |
| -------- | ---------------------- |
| **Mean** | 135.9                  |
| **Min**  | 30.0 ❌ extremely low   |
| **Max**  | 400.0 ❌ extremely high |

So we definitely have:

* **Unrealistically low** values: `< 60`
* **Unrealistically high** values: `> 200` or especially `> 250`

---

### ✅ What we Can Do

In [10]:
# 1. Detect and Flag Outliers

# Using an acceptable range (say 60 to 200):

# Define acceptable range
valid_range = (60, 200)

# Flag outliers
outliers = df[(df['BloodPressure'] < valid_range[0]) | (df['BloodPressure'] > valid_range[1])]
print(outliers)

         Name  Age  BloodPressure Diagnosis
56    charlie   55          400.0   OBESITY
454     alice   68          300.0   obesity
515     diana   82          400.0  Diabetes
643     FRANK   83          300.0    Asthma
899    Hannah   42          400.0   Obesity
1006    alice   81           30.0   HEALTHY
1054    Alice   26          400.0  Diabetes
1114  Charlie   45          400.0   HEALTHY
1489    FRANK   62          300.0  DIABETES
1785   HANNAH   77           30.0   OBESITY


#2. Handle Outliers

Option 1: Remove them
df = df[(df['BloodPressure'] >= 60) & (df['BloodPressure'] <= 200)]
Option 2: Impute with median or cap values
# Cap extreme values
df['BloodPressure'] = df['BloodPressure'].clip(lower=60, upper=200)

In [13]:
# Cap extreme values
df['BloodPressure'] = df['BloodPressure'].clip(lower=60, upper=200)

In [14]:
# 1. Detect and Flag Outliers

# Using an acceptable range (say 60 to 200):

# Define acceptable range
valid_range = (60, 200)

# Flag outliers
outliers = df[(df['BloodPressure'] < valid_range[0]) | (df['BloodPressure'] > valid_range[1])]
print(outliers)

Empty DataFrame
Columns: [Name, Age, BloodPressure, Diagnosis]
Index: []


In [15]:
print(df.head())       # Preview first few rows

      Name  Age  BloodPressure     Diagnosis
0   Hannah   23          153.0  HYPERTENSION
1    FRANK   49          128.0        asthma
2    Diana   19          163.0       Healthy
3  charlie   74          135.0      Diabetes
4    FRANK   85           94.0       Obesity


#### 🧊 3. **Check for Missing Values**

```python
print(df.isnull().sum())
```

✅ **What to Look For**:

* Any column with non-zero missing count needs fixing

In [16]:
print(df.isnull().sum())

Name               0
Age                0
BloodPressure    213
Diagnosis          0
dtype: int64


We're looking at **213 missing values** in the `BloodPressure` column — that’s about **10.7%** of the 1995 total entries.

---

### 🛠️ **What Can We Do with Missing Values?**

We’ve got a few options:

---

### ✅ **Option 1: Impute (Fill in the Gaps)**

#### **1. Fill with the Median**

This is the most common, especially for numeric medical data:

```python
df['BloodPressure'].fillna(df['BloodPressure'].median(), inplace=True)
```

* ✔ Robust to outliers
* ✔ Keeps all rows

#### **2. Fill with Mean**

```python
df['BloodPressure'].fillna(df['BloodPressure'].mean(), inplace=True)
```

#### **3. Fill Using Age Groups or Diagnosis**

If blood pressure correlates with age or diagnosis, we can group by those:

```python
df['BloodPressure'] = df.groupby('Age')['BloodPressure'].transform(lambda x: x.fillna(x.median()))
```

or:

```python
df['BloodPressure'] = df.groupby('Diagnosis')['BloodPressure'].transform(lambda x: x.fillna(x.median()))
```


In [18]:
df['BloodPressure'] = df['BloodPressure'].fillna(df['BloodPressure'].median())
print(df.isnull().sum())


Name             0
Age              0
BloodPressure    0
Diagnosis        0
dtype: int64


#### 📛 4. **Check for Duplicates**

```python
print(df.duplicated().sum())
```

✅ **What to Look For**:

* If >0 → You likely have repeated rows you should remove

In [19]:
print(df.duplicated().sum())


12


Getting `12` as the output means:

> ✅ **There are 12 completely duplicate rows in your DataFrame.**

### ✅ What We Can Do Next

#### 1. **View the Duplicates**

```python
print(df[df.duplicated()])
```

#### 2. **Drop the Duplicates**

```python
df = df.drop_duplicates()
```

#### 3. **Confirm They’re Gone**

```python
print(df.duplicated().sum())  # should now return 0
```

In [20]:
# 1. View the Duplicates

print(df[df.duplicated()])

# 2. Drop the Duplicates

df = df.drop_duplicates()

# 3. Confirm They’re Gone

print(df.duplicated().sum())  # should now return 0

        Name  Age  BloodPressure     Diagnosis
272    DIANA   18          129.0  hypertension
540    ALICE   42          135.0  Hypertension
542   Hannah   36          174.0       obesity
554      eve   85          136.0       Obesity
584    Frank   55          161.0       obesity
763    alice   32          150.0       Healthy
766    Grace   47          170.0       Obesity
915    frank   74          164.0        asthma
969      Ivy   44          135.0  Hypertension
1166     eve   21          124.0      Diabetes
1191    jack   76          165.0  HYPERTENSION
1285     EVE   55           98.0       obesity
0


#### 🔡 5. **Check for Inconsistent Formatting / Casing / Whitespace**

```python
print(df['Name'].unique())
```

✅ **What to Look For**:

* Same name spelled differently (e.g., "Bob", "bob ", "BOB")
* Extra spaces (`'Jack '`, `' jack'`)

In [21]:
print(df['Name'].unique())

['Hannah' 'FRANK' 'Diana' 'charlie' 'Charlie' 'Bob' 'EVE' 'alice' 'eve'
 'ALICE' 'grace' 'ivy' 'Alice' 'HANNAH' 'Jack' 'diana' 'Grace' 'frank'
 'Eve' 'Frank' 'Ivy' 'GRACE' 'hannah' 'JACK' 'bob' 'DIANA' 'frannk' 'BOB'
 'CHARLIE' 'IVY' 'jack' 'Iyv' 'AALICE' 'BBOB' 'JAKC' 'ffrank' 'ccharlie'
 'farnk' 'DDiana' 'RFANK' 'AJCK' 'aliice' 'diiana' 'Hannnah' 'Chharlie'
 'frnak' 'frrank' 'FRAKN' 'allice' 'ALCIE' 'Jacck' 'hnanah' 'FRAANK'
 'jakc' 'CCHARLIE' 'AAlice' 'eeve' 'IIVY' 'rfank']


## ✅ What’s Wrong

From the output, we can see:

### 1. **Casing Issues**

* `'Bob'`, `'BOB'`, `'bob'`
* `'Alice'`, `'ALICE'`, `'alice'`, etc.

### 2. **Misspellings / Variants**

* `'frannk'`, `'frrank'`, `'frnak'`, `'FRAKN'`, `'ffrank'` → probably all meant to be `'Frank'`
* `'diiana'`, `'DDiana'`, `'DIANA'` → likely `'Diana'`
* `'Jack'`, `'jack'`, `'JAKC'`, `'AJCK'`, `'Jacck'` → probably `'Jack'`
* `'charlie'`, `'ccharlie'`, `'CCHARLIE'`, `'Chharlie'` → likely `'Charlie'`

### 3. **Typos / Extra Characters**

* `'AALICE'`, `'aliice'`, `'allice'`, `'AAlice'`, `'ALCIE'` → probably `'Alice'`
* `'Iyv'`, `'IIVY'` → probably `'Ivy'`

---


In [22]:
# Step 1: Strip Whitespace and Standardize Case

df['Name'] = df['Name'].str.strip().str.title()

# Step 2: Manually Correct Misspellings

name_corrections = {
    'Frrank': 'Frank',
    'Frnak': 'Frank',
    'Frannk': 'Frank',
    'Ffrrank': 'Frank',
    'Fraank': 'Frank',
    'Frakn': 'Frank',
    'Rfank': 'Frank',
    'Ffrank': 'Frank',

    'Diiana': 'Diana',
    'Ddiana': 'Diana',
    'Diana ': 'Diana',

    'Chharlie': 'Charlie',
    'Ccharlie': 'Charlie',
    'Ccharlie ': 'Charlie',

    'Jakc': 'Jack',
    'Jakc': 'Jack',
    'Ajck': 'Jack',
    'Jacck': 'Jack',

    'Aliice': 'Alice',
    'Allice': 'Alice',
    'Aalice': 'Alice',
    'Alcie': 'Alice',
    'Aaice': 'Alice',

    'Iyv': 'Ivy',
    'Iivy': 'Ivy',
    'Iiivy': 'Ivy',
}

df['Name'] = df['Name'].replace(name_corrections)

# Step 3: Recheck Uniques
print(sorted(df['Name'].unique()))



['Alice', 'Bbob', 'Bob', 'Charlie', 'Diana', 'Eeve', 'Eve', 'Farnk', 'Frank', 'Grace', 'Hannah', 'Hannnah', 'Hnanah', 'Ivy', 'Jack']


The list is now shorter, but there are still some **stragglers and misspellings** that should be cleaned up.

We will go through them:

---

### ❌ Remaining Misspellings / Inconsistencies:

| Incorrect | Intended |
| --------- | -------- |
| `Bbob`    | `Bob`    |
| `Eeve`    | `Eve`    |
| `Farnk`   | `Frank`  |
| `Hannnah` | `Hannah` |
| `Hnanah`  | `Hannah` |

---

### ✅ Final Fix

We can use this small mapping dictionary to correct them:

In [23]:
name_fixes = {
    'Bbob': 'Bob',
    'Eeve': 'Eve',
    'Farnk': 'Frank',
    'Hannnah': 'Hannah',
    'Hnanah': 'Hannah'
}

df['Name'] = df['Name'].replace(name_fixes)

# Then recheck:

print(sorted(df['Name'].unique()))

['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Hannah', 'Ivy', 'Jack']


Use `.value_counts()` to inspect frequency:

```python
print(df['Diagnosis'].value_counts(dropna=False))
```

In [24]:
print(df['Diagnosis'].value_counts(dropna=False))

Diagnosis
DIABETES        166
diabetes        149
Obesity         148
Diabetes        146
Asthma          135
HEALTHY         133
OBESITY         131
healthy         130
Healthy         128
hypertension    127
ASTHMA          124
obesity         119
HYPERTENSION    118
Hypertension    117
asthma          112
Name: count, dtype: int64


The output makes it very clear: the `'Diagnosis'` column suffers from **inconsistent casing and formatting**, just like `'Name'` earlier.

---

### 🔍 Issues Identified:


We have the same diagnoses written in many different ways:

| Diagnosis      | Variants Found                                       |
| -------------- | ---------------------------------------------------- |
| `Diabetes`     | `'diabetes'`, `'Diabetes'`, `'DIABETES'`             |
| `Obesity`      | `'Obesity'`, `'OBESITY'`, `'obesity'`                |
| `Asthma`       | `'Asthma'`, `'ASTHMA'`, `'asthma'`                   |
| `Hypertension` | `'Hypertension'`, `'HYPERTENSION'`, `'hypertension'` |
| `Healthy`      | `'Healthy'`, `'HEALTHY'`, `'healthy'`                |

### ✅ Step-by-Step Fix

#### 1. Standardize Casing

```python
df['Diagnosis'] = df['Diagnosis'].str.strip().str.title()
```

This will convert all variants to a consistent format:

* `'DIABETES'`, `'diabetes'` → `'Diabetes'`
* `'HEALTHY'`, `'healthy'` → `'Healthy'`

---

#### 2. Recheck the Counts

```python
print(df['Diagnosis'].value_counts())
```

In [25]:
# 1. Standardize Casing

df['Diagnosis'] = df['Diagnosis'].str.strip().str.title()

# 2. Recheck the Counts

print(df['Diagnosis'].value_counts())

Diagnosis
Diabetes        461
Obesity         398
Healthy         391
Asthma          371
Hypertension    362
Name: count, dtype: int64


### ✅ . **Save Cleaned Dataset**

```python
df.to_csv("patients_cleaned.csv", index=False)
```

In [27]:
df.to_csv("C:\\Users\\frank\\data_science_project_advance\\datasets\\patients_cleaned.csv", index=False)